# AI-Face Phase E: Global / Local / Forensic fusion
Attach two **full-run artifacts**, not the image dataset: (1) notebook 11's `facetrace_spectral_artifacts.zip` uploaded as a private dataset, or its saved Notebook Output; (2) the teammate's saved Global/Local Notebook Output, or a ZIP of its complete `outputs/aiface_phase_e` directory.

Enable Internet; Accelerator = None (CPU). This notebook clones FaceTrace main. No HF_TOKEN, DINOv3 extraction, FFT extraction or training is needed.

Fusion is equal probability averaging: Local+Forensic and Global+Local+Forensic. Thresholds are selected on validation only, then locked for test metrics and rescue/harm. Artifacts must share manifest checksums, generator split, seed and image size. Smoke runs and old Who Is AI outputs are rejected.

The current Global/Local runner does not export validation predictions. Its full output must include `feature_cache.pt` and `global_only/checkpoint.pt`, `local_only/checkpoint.pt` (or the corresponding `predictions_val.csv` files). Test CSVs alone are insufficient.

In [ ]:
from pathlib import Path
import os, subprocess, sys, json, shutil

REPO_URL = 'https://github.com/AIVIETNAM-AIO-DinhBao/FaceTrace.git'
REPO_REF = 'main'  # Pin a commit for exact reproduction after the first successful run
REPO_DIR = Path('/kaggle/working/FaceTrace')
FORENSIC_INPUT = None  # Optional exact /kaggle/input directory or artifact ZIP path
BASELINE_INPUT = None  # Optional exact /kaggle/input directory or artifact ZIP path
OUT = Path('/kaggle/working/outputs/aiface_phase_e_fusion')
OUT.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('OMP_NUM_THREADS', '2')
os.environ.setdefault('MKL_NUM_THREADS', '2')
for p in sorted(Path('/kaggle/input').iterdir()):
    print('Attached input:', p)


In [ ]:
if REPO_DIR.exists() and any(REPO_DIR.iterdir()):
    raise RuntimeError('Restart the Kaggle session to clone a fresh source version.')
subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(['git', 'checkout', '--detach', REPO_REF], cwd=REPO_DIR, check=True)
assert (REPO_DIR / 'scripts/run_aiface_fusion_kaggle.py').is_file(), 'Source is missing the fusion runner'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'PyYAML>=6', 'pandas>=2.2', 'scikit-learn>=1.4', 'Pillow>=10', 'tqdm>=4.66'], check=True)
import torch, pandas as pd
from IPython.display import display, FileLink
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
print('Source commit:', commit)
print('Fusion device: CPU; torch:', torch.__version__)
(OUT / 'notebook_session.json').write_text(json.dumps({'git_commit': commit, 'device': 'cpu', 'forensic_input': str(FORENSIC_INPUT), 'baseline_input': str(BASELINE_INPUT)}, indent=2))


In [ ]:
def run_logged(command, log_name):
    with (OUT / log_name).open('w') as log:
        process = subprocess.Popen(command, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='')
            log.write(line); log.flush()
        code = process.wait()
    if code:
        raise RuntimeError(f'Command failed ({code}); see {OUT / log_name}')

run_logged([sys.executable, '-u', '-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_fusion_inputs.py', '-v'], 'input_tests.log')
command = [sys.executable, '-u', str(REPO_DIR / 'scripts/run_aiface_fusion_kaggle.py'), '--output-dir', str(OUT)]
if FORENSIC_INPUT:
    command.extend(['--forensic-input', str(FORENSIC_INPUT)])
if BASELINE_INPUT:
    command.extend(['--baseline-input', str(BASELINE_INPUT)])
run_logged(command, 'fusion.log')


In [ ]:
inputs = json.loads((OUT / 'fusion_inputs.json').read_text())
baseline = Path(inputs['baseline']['root'])
forensic = Path(inputs['forensic']['root'])
models = {'global_only': baseline / 'global_only', 'local_only': baseline / 'local_only', 'global_local': baseline / 'global_local', 'forensic_only': forensic, 'local_forensic': OUT / 'local_forensic', 'global_local_forensic': OUT / 'global_local_forensic'}
rows, generator_rows = [], []
for name, directory in models.items():
    metrics = json.loads((directory / 'test_metrics_overall.json').read_text())
    threshold = json.loads((directory / 'val_metrics.json').read_text())['threshold']
    rows.append({'model': name, 'macro_auroc': metrics['macro']['auroc'], 'macro_balanced_accuracy': metrics['macro']['balanced_accuracy'], 'overall_accuracy': metrics['accuracy'], 'overall_f1': metrics['f1'], 'val_threshold': threshold})
    per_generator = pd.read_csv(directory / 'test_metrics_by_generator.csv')
    per_generator.insert(0, 'model', name)
    generator_rows.append(per_generator)
comparison = pd.DataFrame(rows)
comparison.to_csv(OUT / 'comparison.csv', index=False)
generator_comparison = pd.concat(generator_rows, ignore_index=True)
generator_comparison.to_csv(OUT / 'comparison_by_generator.csv', index=False)
display(comparison)
display(generator_comparison)
for name in ['local_forensic', 'global_local_forensic']:
    print(name, ': rescue/harm vs Local-only or Global+Local; each model uses its locked validation threshold')
    display(pd.read_csv(OUT / name / 'rescue_harm.csv'))
print('Positive net_gain = more rescued errors than newly harmed predictions. Fusion is not guaranteed to improve every metric.')


In [ ]:
# Keep the compact evidence used for comparisons, not the large feature caches.
for name, directory in models.items():
    if directory.parent == OUT:
        continue
    destination = OUT / 'input_evidence' / name
    destination.mkdir(parents=True, exist_ok=True)
    for filename in ['run_metadata.json', 'val_metrics.json', 'test_metrics_overall.json', 'test_metrics_by_generator.csv', 'predictions_val.csv', 'predictions_test.csv', 'environment.json', 'config_resolved.yaml']:
        if (directory / filename).is_file():
            shutil.copy2(directory / filename, destination / filename)
shutil.copy2(baseline / 'run_metadata.json', OUT / 'input_evidence/baseline_run_metadata.json')
for directory in ['src', 'scripts', 'configs', 'tests']:
    shutil.copytree(REPO_DIR / directory, OUT / 'source' / directory, dirs_exist_ok=True, ignore=shutil.ignore_patterns('__pycache__'))
archive = shutil.make_archive('/kaggle/working/facetrace_fusion_artifacts', 'zip', root_dir=OUT)
print('Download:', archive)
display(FileLink(archive))
print('Save Version -> Save & Run All for the final saved run. Keep its URL and both attached artifact versions.')
